In [1]:
import os
# Use the repository root as the working directory, whether this notebook is
# launched from the repo root or from the notebooks/ folder.
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')


# Notebook 09: Bootstrap Confidence Intervals

1,000 resamples, percentile method (2.5 and 97.5), seed 42.  
Applied to all metrics × all models × both datasets.

In [2]:
import numpy as np
import pandas as pd
from sklearn.metrics import (
    f1_score, accuracy_score, roc_auc_score,
    matthews_corrcoef, balanced_accuracy_score,
    average_precision_score, precision_score, recall_score,
    confusion_matrix
)
import warnings
warnings.filterwarnings('ignore')

N_BOOT = 1000
SEED   = 42
rng    = np.random.default_rng(SEED)
print('Libraries loaded.')

Libraries loaded.


In [3]:
def compute_metrics(y_true, y_pred, y_prob):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0
    npv  = tn / (tn + fn) if (tn + fn) > 0 else 0
    fpr  = fp / (fp + tn) if (fp + tn) > 0 else 0
    return {
        'f1_macro':     f1_score(y_true, y_pred, average='macro', zero_division=0),
        'f1_attack':    f1_score(y_true, y_pred, pos_label=1, zero_division=0),
        'f1_benign':    f1_score(y_true, y_pred, pos_label=0, zero_division=0),
        'precision':    precision_score(y_true, y_pred, zero_division=0),
        'recall':       recall_score(y_true, y_pred, zero_division=0),
        'accuracy':     accuracy_score(y_true, y_pred),
        'auc_roc':      roc_auc_score(y_true, y_prob),
        'pr_auc':       average_precision_score(y_true, y_prob),
        'mcc':          matthews_corrcoef(y_true, y_pred),
        'balanced_acc': balanced_accuracy_score(y_true, y_pred),
        'specificity':  spec,
        'npv':          npv,
        'fpr':          fpr,
    }

def bootstrap_ci(y_true, y_pred, y_prob, n_boot=N_BOOT, seed=SEED):
    rng_b = np.random.default_rng(seed)
    n = len(y_true)
    boot_metrics = []
    for _ in range(n_boot):
        idx = rng_b.integers(0, n, size=n)
        # Skip if only one class present
        if len(np.unique(y_true[idx])) < 2:
            continue
        boot_metrics.append(compute_metrics(y_true[idx], y_pred[idx], y_prob[idx]))
    result = {}
    for key in boot_metrics[0]:
        vals = np.array([m[key] for m in boot_metrics])
        result[f'{key}_lo'] = round(float(np.percentile(vals, 2.5)), 6)
        result[f'{key}_hi'] = round(float(np.percentile(vals, 97.5)), 6)
    return result

print('Functions defined.')

Functions defined.


In [4]:
MODELS   = ['RandomForest', 'DecisionTree', 'XGBoost', 'LogisticRegression']
DATASETS = [('UGRansome2024', 'ugr'), ('CICIoT2023', 'cic')]
all_rows = []

for dataset, prefix in DATASETS:
    for model in MODELS:
        print(f'{dataset} | {model} ...', end=' ')
        df = pd.read_csv(f'results/baselines/{prefix}_{model}_predictions.csv')
        y_true = df['y_true'].values
        y_pred = df['y_pred'].values
        y_prob = df['y_prob'].values

        point = compute_metrics(y_true, y_pred, y_prob)
        ci    = bootstrap_ci(y_true, y_pred, y_prob)

        row = {'dataset': dataset, 'model': model}
        row.update({k: round(v, 6) for k, v in point.items()})
        row.update(ci)
        all_rows.append(row)
        print(f"F1={point['f1_macro']:.4f} [{ci['f1_macro_lo']:.4f}, {ci['f1_macro_hi']:.4f}]")

df_ci = pd.DataFrame(all_rows)
df_ci.to_csv('results/bootstrap_ci.csv', index=False)
print('\nSaved results/bootstrap_ci.csv')
df_ci[['dataset','model','f1_macro','f1_macro_lo','f1_macro_hi','auc_roc','auc_roc_lo','auc_roc_hi']]

UGRansome2024 | RandomForest ... 

F1=0.9911 [0.9895, 0.9927]
UGRansome2024 | DecisionTree ... 

F1=0.9927 [0.9913, 0.9941]
UGRansome2024 | XGBoost ... 

F1=0.9944 [0.9932, 0.9957]
UGRansome2024 | LogisticRegression ... 

F1=0.8951 [0.8898, 0.8999]
CICIoT2023 | RandomForest ... 

F1=0.9622 [0.9556, 0.9679]
CICIoT2023 | DecisionTree ... 

F1=0.9493 [0.9421, 0.9565]
CICIoT2023 | XGBoost ... 

F1=0.9464 [0.9395, 0.9532]
CICIoT2023 | LogisticRegression ... 

F1=0.8283 [0.8180, 0.8389]

Saved results/bootstrap_ci.csv


,dataset,model,f1_macro,f1_macro_lo,f1_macro_hi,auc_roc,auc_roc_lo,auc_roc_hi
0,UGRansome2024,RandomForest,0.991080,0.989538,0.992694,0.999857,0.999816,0.999895
1,UGRansome2024,DecisionTree,0.992731,0.991325,0.994111,0.993218,0.991657,0.994682
2,UGRansome2024,XGBoost,0.994416,0.993178,0.995667,0.999939,0.999918,0.999959
3,UGRansome2024,LogisticRegression,0.895109,0.889822,0.899933,0.948168,0.944687,0.951113
4,CICIoT2023,RandomForest,0.962162,0.955639,0.967918,0.999530,0.999402,0.999644
5,CICIoT2023,DecisionTree,0.949307,0.942144,0.956522,0.941429,0.931188,0.951895
6,CICIoT2023,XGBoost,0.946426,0.939541,0.953246,0.999349,0.999198,0.999497
7,CICIoT2023,LogisticRegression,0.828337,0.818017,0.838926,0.985447,0.980159,0.990299


In [5]:
print('Notebook 09 complete.')

Notebook 09 complete.
